**Build a customer analytics and segmentation dataset that summarizes each customer’s order behavior, spending, and recency, and classifies them into value segments for marketing, loyalty, and retention strategies.**


In [0]:
analytics = spark.sql("""
WITH agg AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(net_amount) AS net_sales_amount,
        SUM(usd_net_amount) AS usd_net_sales_amount,
        AVG(net_amount) AS avg_order_value,
        MAX(order_ts_utc) AS last_order_ts
    FROM globalretail.silver.fact_sales
    GROUP BY customer_id
)

SELECT
    a.*,
    c.customer_sk,
    c.full_name,
    c.city,
    c.country_code,
    c.loyalty_tier,
    CASE
        WHEN a.usd_net_sales_amount >= 1000 THEN 'HIGH_VALUE'
        WHEN a.usd_net_sales_amount >= 300 THEN 'MID_VALUE'
        ELSE 'EMERGING'
    END AS customer_segment
FROM agg a
LEFT JOIN (
    SELECT
        customer_id,
        customer_sk,
        full_name,
        city,
        country_code,
        loyalty_tier
    FROM globalretail.silver.dim_customers
) c
ON a.customer_id = c.customer_id
""")

In [0]:
analytics.write.mode("overwrite").saveAsTable("globalretail.gold.customer_analytics")